# Proyecto 2 · Sistema integrado de clasificación, segmentación y atención
## Notebook 01 — Clasificación de lesiones dermatoscópicas con y sin CBAM

**Curso:** Visión Computacional con Deep Learning — Maestría en Inteligencia Artificial y Ciencia de Datos, UAO

**Integrantes:** Maria Stella Fuentes Diaz · Sergio Luis Castaño Rodríguez · Alejandro Galvez Cardenas · Joan Sebastian Mena Ortega

## Objetivo

Ajustar mediante *fine-tuning* una ResNet-34 preentrenada en ImageNet para clasificar
imágenes dermatoscópicas en siete categorías diagnósticas; cuantificar el aporte de un
módulo de atención CBAM integrado en el backbone, mediante una comparación controlada con
y sin el módulo; interpretar las predicciones con Grad-CAM, y documentar la optimización
del modelo y sus tiempos de inferencia.

## Problema y datos

La dermatoscopia es una técnica de imagen no invasiva que, mediante aumento óptico e
iluminación controlada, hace visibles estructuras pigmentadas de la epidermis y la dermis
superficial que no se aprecian a simple vista. Su interpretación exige entrenamiento y
presenta variabilidad entre observadores, lo que motiva sistemas de apoyo para el
**triaje** de lesiones pigmentadas: identificar qué lesiones requieren valoración
prioritaria por un especialista.

La clasificación automática es difícil por dos razones. Primero, lesiones benignas y
malignas pueden compartir rasgos visuales; la confusión más relevante es entre nevus y
melanoma. Segundo, la distribución de diagnósticos es muy desbalanceada, como en la
práctica clínica, donde predominan las lesiones benignas.

Se utiliza **HAM10000** (*Human Against Machine with 10000 training images*; Tschandl
et al., 2018), el conjunto de entrenamiento de la Tarea 3 del challenge ISIC 2018. Reúne
**10.015 imágenes dermatoscópicas** de 600×450 píxeles, correspondientes a **7.470
lesiones** distintas: una misma lesión puede aparecer en varias fotografías. Las imágenes
provienen de dos centros, el Departamento de Dermatología de la Universidad Médica de
Viena (Austria) y una consulta especializada en cáncer de piel en Queensland (Australia).

El diagnóstico de referencia se confirmó por **histopatología en el 53,3%** de las
imágenes; el resto, por seguimiento clínico (37,0%), consenso de expertos (9,0%) o
microscopía confocal (0,7%). En las tres clases malignas o premalignas (`mel`, `bcc`,
`akiec`) la confirmación histopatológica es del 100%. Cada imagen trae además una
**máscara binaria de la lesión**, generada de forma semiautomática y revisada
manualmente, lo que permite resolver clasificación y segmentación sobre las mismas
imágenes.

| Clase | Diagnóstico | Imágenes | % |
| --- | --- | ---: | ---: |
| `nv` | Nevus melanocítico | 6.705 | 66,9 |
| `mel` | Melanoma | 1.113 | 11,1 |
| `bkl` | Lesión benigna tipo queratosis | 1.099 | 11,0 |
| `bcc` | Carcinoma basocelular | 514 | 5,1 |
| `akiec` | Queratosis actínica / carcinoma intraepitelial | 327 | 3,3 |
| `vasc` | Lesión vascular | 142 | 1,4 |
| `df` | Dermatofibroma | 115 | 1,1 |

- **Repositorio:** Harvard Dataverse, DOI [10.7910/DVN/DBW86T](https://doi.org/10.7910/DVN/DBW86T) (licencia CC BY-NC 4.0)
- **Máscaras:** archivo `HAM10000_segmentations_lesion_tschandl.zip` del mismo repositorio

El alcance de este notebook es la **clasificación de la imagen completa**, que contiene
una única lesión. La segmentación y el mecanismo de self-attention se desarrollan en el
notebook 02. El sistema es una herramienta académica de apoyo, **no un dispositivo
diagnóstico**.

## Guía del notebook

Ninguna decisión de modelado se toma antes de conocer los datos: la sección 1 caracteriza
el conjunto y la partición; las secciones 2 y 3 definen y entrenan el modelo; las
secciones 4 a 7 evalúan, interpretan, optimizan y miden tiempos. La sección 0 solo
prepara el entorno de ejecución.

| Qué buscar | Dónde está |
| --- | --- |
| Partición por lesión, desbalance y ejemplos por clase | Sección 1 |
| Justificación de la métrica principal (macro-F1) | Sección 1.4 |
| Arquitectura: ResNet-34 y CBAM (fórmulas, implementación y verificación) | Sección 2 |
| Pérdida ponderada, ciclo de entrenamiento y entrenamiento con y sin CBAM | Sección 3 |
| Curvas de pérdida de entrenamiento y validación | Sección 3.5 |
| Resultados en test: tabla, prueba de significancia, matrices de confusión y F1 por clase | Sección 4 |
| Grad-CAM: implementación, verificación y comparación con y sin CBAM | Sección 5 |
| Cuantización INT8: tamaño y métricas antes y después | Sección 6 |
| Tiempos de inferencia en GPU | Sección 7 |
| Checklist de verificación | Sección 8 |

**Antes de ejecutar:** seleccionar *Entorno de ejecución → Cambiar tipo de entorno → GPU
T4*. Duración estimada: unas 2 horas. Si Colab
se desconecta, basta con volver a ejecutar desde arriba: los entrenamientos que ya
terminaron no se repiten, y uno interrumpido continúa desde la última época completada.

## 0. Preparación del entorno

Esta sección deja lista la máquina de Colab: verifica la GPU, conecta Google Drive,
descarga el código y el dataset, y configura las rutas. Se ejecuta una vez por sesión y
es **reanudable**: si Colab se desconecta, basta con volver a ejecutarla desde arriba.

### 0.1 Verificar la GPU

El entrenamiento requiere una GPU. Si la sesión no la tiene, la celda se detiene con
instrucciones para activarla.

In [ ]:
import sys
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'No hay GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> GPU T4, '
        'y vuelve a ejecutar esta celda.'
    )

print('GPU    :', torch.cuda.get_device_name(0))
print('VRAM   :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('torch  :', torch.__version__)
print('CUDA   :', torch.version.cuda)
print('Python :', sys.version.split()[0])

### 0.2 Rutas de trabajo

La máquina de Colab se borra al desconectarse, así que se separan dos ubicaciones:

| Qué | Dónde | Por qué |
| --- | --- | --- |
| Dataset (2,6 GB) | `/content/data`, disco de la sesión | El entrenamiento lee cada imagen miles de veces; desde Drive sería mucho más lento |
| Modelos, métricas y figuras | Google Drive | Se guardan desde la primera época, así que una desconexión no borra el trabajo |

In [ ]:
from pathlib import Path

RAIZ_DATOS = Path('/content/data')                        # dataset, disco local de la sesion
RAIZ_DRIVE = Path('/content/drive/MyDrive/dermascope')    # modelos, metricas y figuras
REPO_DIR = Path('/content/dermascope')                    # codigo clonado de GitHub

### 0.3 Conectar Google Drive

Se monta Drive y se crean las carpetas donde quedarán los resultados. Colab pedirá
autorización para acceder a la cuenta.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
for sub in ('models', 'reports/results', 'reports/figures'):
    (RAIZ_DRIVE / sub).mkdir(parents=True, exist_ok=True)
print('Drive listo en', RAIZ_DRIVE)

### 0.4 Descargar el código

Se clona el repositorio público del proyecto; si ya estaba clonado, solo se actualiza
con la última versión. Si la carpeta existe pero está vacía o incompleta (por ejemplo,
por una descarga interrumpida), se descarta y se vuelve a clonar.

In [ ]:
REPO_URL = 'https://github.com/mariastella1807/dermascope.git'

import os
import shutil
import subprocess


def clon_valido(ruta):
    """True si la carpeta es un repositorio git con al menos un commit."""
    resultado = subprocess.run(['git', '-C', str(ruta), 'rev-parse', 'HEAD'], capture_output=True)
    return (ruta / '.git').exists() and resultado.returncode == 0


if REPO_DIR.exists() and not clon_valido(REPO_DIR):
    os.chdir('/content')
    shutil.rmtree(REPO_DIR)

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('directorio:', Path.cwd())
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

### 0.5 Dependencias

Colab ya incluye las librerías que usa el proyecto (PyTorch, torchvision, scikit-learn,
scipy, pandas y matplotlib). Solo se instalan las de exportación a ONNX, necesarias para
la cuantización. A propósito **no** se instala desde `requirements.txt`: reinstalar
PyTorch sobre el de Colab rompe su compatibilidad con la GPU.

In [ ]:
!pip install --quiet onnx onnxruntime onnxscript

### 0.6 Verificación del entorno

Antes de gastar tiempo de GPU se ejecutan las comprobaciones del proyecto
(`scripts/smoke_test.py`): construyen los modelos, prueban las métricas y verifican que
el proyecto no dependa de librerías innecesarias. No necesitan el dataset. Si alguna falla, el
problema es de versiones y conviene resolverlo aquí.

In [ ]:
!python scripts/smoke_test.py

### 0.7 Descargar el dataset

Se descargan de Harvard Dataverse (DOI `10.7910/DVN/DBW86T`) los metadatos, las
imágenes y las máscaras. La descarga es anónima y toma unos minutos; los archivos que ya
existen se saltan.

In [ ]:
%%bash
set -e
RAW=/content/data/raw
mkdir -p $RAW

dl() {  # dl <id> <destino>
  if [ -s "$2" ]; then echo "ya existe: $(basename $2)"; return; fi
  curl -sS -L --retry 5 --retry-delay 3 -C - \
    -o "$2" "https://dataverse.harvard.edu/api/access/datafile/$1"
  echo "$(basename $2): $(du -h $2 | cut -f1)"
}

dl 4338392 $RAW/HAM10000_metadata.tab
dl 3838943 $RAW/segmentations.zip
dl 3172585 $RAW/HAM10000_images_part_1.zip
dl 3172584 $RAW/HAM10000_images_part_2.zip

ls -lh $RAW

### 0.8 Descomprimir

Se extraen las imágenes y las máscaras. La extracción es reanudable: salta los archivos
que ya están completos.

In [ ]:
!python scripts/extract_data.py /content/data/raw/HAM10000_images \
    /content/data/raw/HAM10000_images_part_1.zip \
    /content/data/raw/HAM10000_images_part_2.zip

!python scripts/extract_data.py /content/data/raw/HAM10000_segmentations \
    /content/data/raw/segmentations.zip

### 0.9 Configuración de rutas

El código lee las rutas de `configs/paths.yaml`. Este archivo local las ajusta a Colab
sin modificar el repositorio (los `configs/*.local.yaml` no se versionan).

In [ ]:
Path('configs/paths.local.yaml').write_text(
    'paths:\n'
    f'  ham_metadata: {RAIZ_DATOS.as_posix()}/raw/HAM10000_metadata.tab\n'
    f'  ham_images: {RAIZ_DATOS.as_posix()}/raw/HAM10000_images\n'
    f'  seg_masks: {RAIZ_DATOS.as_posix()}/raw/HAM10000_segmentations\n'
    f'  processed: {RAIZ_DATOS.as_posix()}/processed\n'
    f'  models: {(RAIZ_DRIVE / "models").as_posix()}\n'
    f'  reports: {(RAIZ_DRIVE / "reports").as_posix()}\n',
    encoding='utf-8',
)
print(Path('configs/paths.local.yaml').read_text(encoding='utf-8'))

### 0.10 Utilidades del notebook

Importaciones comunes, las carpetas de resultados en Drive, el dispositivo de cómputo y
dos funciones auxiliares: una guarda cada figura para el informe y otra lee los
resultados que escriben los scripts de entrenamiento.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.config import load_config, resolve

DIR_MODELOS = RAIZ_DRIVE / 'models'
DIR_RESULTADOS = RAIZ_DRIVE / 'reports' / 'results'
DIR_FIGURAS = RAIZ_DRIVE / 'reports' / 'figures'
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def guardar_figura(fig, nombre):
    """Guarda la figura en Drive para usarla en el informe."""
    ruta = DIR_FIGURAS / f'{nombre}.png'
    fig.savefig(ruta, dpi=150, bbox_inches='tight')
    print('figura guardada en', ruta)


def leer_json(nombre):
    return json.loads((DIR_RESULTADOS / nombre).read_text(encoding='utf-8'))


print('dispositivo:', DISPOSITIVO)

## 1. Exploración del dataset

Antes de modelar se construye y verifica la partición de los datos, se cuantifica el
desbalance de clases, se observan ejemplos de cada clase y se justifica la métrica de
evaluación.

### 1.1 Partición por lesión, no por imagen

Como una misma lesión puede aparecer en varias fotografías, repartir las imágenes al azar
pondría fotos de una misma lesión en entrenamiento y en test: el modelo la *reconocería*
en vez de generalizar, y la métrica de test quedaría inflada. Por eso la partición
70/15/15 se hace sobre `lesion_id`, estratificada por diagnóstico y con semilla fija.

In [ ]:
!python -m src.data.build_splits --config configs/paths.yaml

Verificación de la partición: número de imágenes y de lesiones por split, y comprobación
de que ninguna lesión quedó repartida entre dos splits.

In [ ]:
cfg_cls = load_config('configs/classification.yaml')
CLASES = list(cfg_cls.classes)
NOMBRES_ES = dict(cfg_cls.class_names_es)
splits = pd.read_csv(resolve(cfg_cls.paths.processed) / 'splits.csv')

fotos_por_lesion = splits.groupby('lesion_id').size()
print(f"{len(splits)} imagenes de {splits['lesion_id'].nunique()} lesiones distintas "
      f"({len(splits) / splits['lesion_id'].nunique():.2f} fotos por lesion en promedio)")
print(f"{(fotos_por_lesion > 1).sum()} lesiones tienen mas de una foto\n")

resumen_splits = pd.DataFrame({
    'imagenes': splits['split'].value_counts(),
    'lesiones_unicas': splits.groupby('split')['lesion_id'].nunique(),
}).loc[['train', 'val', 'test']]
print(resumen_splits)

# Verificacion: ninguna lesion puede estar en dos splits
fuga = int((splits.groupby('lesion_id')['split'].nunique() > 1).sum())
assert fuga == 0, f'{fuga} lesiones aparecen en mas de un split'
print('\nverificado: ninguna lesion aparece en mas de un split')

### 1.2 Distribución de clases

Se cuantifica el desbalance por split y en total. Esta distribución condiciona la
métrica de evaluación (sección 1.4) y la pérdida de entrenamiento (sección 3.1).

In [ ]:
conteos = pd.crosstab(splits['dx'], splits['split'])[['train', 'val', 'test']].reindex(CLASES)
conteos['total'] = conteos.sum(axis=1)
conteos['porcentaje'] = (100 * conteos['total'] / conteos['total'].sum()).round(1)
conteos['diagnostico'] = [NOMBRES_ES[c] for c in conteos.index]
orden = conteos.sort_values('total', ascending=False)
print(orden)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(orden.index, orden['total'], color='#4C72B0')
for i, (n, pct) in enumerate(zip(orden['total'], orden['porcentaje'])):
    ax.text(i, n, f'{n}\n({pct}%)', ha='center', va='bottom', fontsize=8)
ax.set_ylim(0, orden['total'].max() * 1.25)
ax.set_ylabel('imagenes')
ax.set_title('Distribucion de clases en HAM10000')
plt.tight_layout()
guardar_figura(fig, 'distribucion_clases')
plt.show()

print(f"\nla clase mayor tiene {orden['total'].max() / orden['total'].min():.0f} veces mas imagenes que la menor")

### 1.3 Ejemplos por clase

Tres imágenes de entrenamiento elegidas al azar por clase, para observar la variabilidad
dentro de cada clase y la similitud visual entre clases distintas.

In [ ]:
from PIL import Image

rng = np.random.default_rng(42)
fig, axes = plt.subplots(3, len(CLASES), figsize=(2.2 * len(CLASES), 6.8))
for j, clase in enumerate(CLASES):
    filas = splits[(splits['split'] == 'train') & (splits['dx'] == clase)]
    for i, idx in enumerate(rng.choice(len(filas), size=3, replace=False)):
        axes[i, j].imshow(Image.open(filas.iloc[idx]['image_path']))
        axes[i, j].axis('off')
    axes[0, j].set_title(clase, fontsize=11)
plt.suptitle('Tres ejemplos por clase (train)')
plt.tight_layout()
guardar_figura(fig, 'ejemplos_por_clase')
plt.show()

for clase in CLASES:
    print(f'{clase:6s} {NOMBRES_ES[clase]}')

### 1.4 Métrica principal: macro-F1

Con `nv` cerca del 67% del dataset, un modelo que responda siempre "nevus" alcanzaría
alrededor de 67% de accuracy sin haber aprendido nada. El F1 de cada clase combina
precisión y recall, y el **macro-F1** promedia las siete clases por igual, de modo que
ignorar las clases minoritarias se penaliza:

$$F1_c = \frac{2\,P_c\,R_c}{P_c + R_c} \qquad\qquad \text{macro-}F1 = \frac{1}{K}\sum_{c=1}^{K} F1_c$$

Se reportan ambas métricas, pero el checkpoint se selecciona por macro-F1 de validación.
La celda siguiente lo comprueba con el conjunto de test real y un clasificador constante.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_test = splits.loc[splits['split'] == 'test', 'dx_idx'].to_numpy()
y_constante = np.full_like(y_test, CLASES.index('nv'))

acc_constante = accuracy_score(y_test, y_constante)
f1_constante = f1_score(y_test, y_constante, average='macro', zero_division=0)
print(f'Clasificador que siempre responde "nv":  accuracy = {acc_constante:.3f}   macro-F1 = {f1_constante:.3f}')

assert acc_constante > 0.6 and f1_constante < 0.2
print('verificado: la accuracy premia al clasificador constante; el macro-F1 no')

## 2. El modelo: ResNet-34 + CBAM

El clasificador parte de una ResNet-34 (He et al., 2016) preentrenada en ImageNet. Se
eligió ResNet porque sus etapas `layer1` a `layer4` son módulos secuenciales, lo que
permite insertar CBAM **dentro** del backbone, al final de una etapa, y no como un módulo
añadido después del pooling global.

### 2.1 Qué hace CBAM

CBAM (*Convolutional Block Attention Module*; Woo et al., 2018) toma un mapa de
características $F \in \mathbb{R}^{C \times H \times W}$ y lo repondera en dos pasos.

**Atención de canal: ¿qué filtros son relevantes?** Resume cada canal con un promedio y
un máximo espaciales, los pasa por un MLP compartido y produce un peso por canal:

$$M_c(F) = \sigma\big(\text{MLP}(\text{AvgPool}(F)) + \text{MLP}(\text{MaxPool}(F))\big) \in \mathbb{R}^{C\times 1\times 1}
\qquad F' = M_c(F) \otimes F$$

Es una extensión del bloque Squeeze-and-Excitation (Hu et al., 2018), que solo usa el
promedio.

**Atención espacial: ¿en qué zona de la imagen?** Resume los canales en cada posición
con un promedio y un máximo, y una convolución 7×7 produce un peso por píxel:

$$M_s(F') = \sigma\big(f^{7\times 7}([\text{AvgPool}_c(F');\ \text{MaxPool}_c(F')])\big) \in \mathbb{R}^{1\times H\times W}
\qquad F'' = M_s(F') \otimes F'$$

En dermatoscopia la rama espacial es especialmente pertinente: la imagen contiene mucha
piel sana y artefactos (vello, burbujas, viñeteado) que conviene descontar.

La celda siguiente implementa CBAM paso a paso, siguiendo las dos fórmulas.

In [ ]:
import torch.nn as nn


class CBAMDidactico(nn.Module):
    '''CBAM escrito paso a paso, siguiendo las dos formulas de arriba.'''

    def __init__(self, canales, reduccion=16, kernel=7):
        super().__init__()
        ocultas = max(canales // reduccion, 8)
        # MLP compartido de la rama de canal: el mismo para el promedio y para el maximo
        self.mlp = nn.Sequential(
            nn.Linear(canales, ocultas, bias=False),
            nn.ReLU(),
            nn.Linear(ocultas, canales, bias=False),
        )
        # Convolucion de la rama espacial: entran 2 mapas (promedio y maximo), sale 1
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel, padding=kernel // 2, bias=False)

    def forward(self, F_entrada):
        B, C, H, W = F_entrada.shape

        # ---- Rama de canal ----
        resumen_avg = F_entrada.mean(dim=(2, 3))                     # (B, C)
        resumen_max = F_entrada.amax(dim=(2, 3))                     # (B, C)
        M_c = torch.sigmoid(self.mlp(resumen_avg) + self.mlp(resumen_max))
        F_prima = F_entrada * M_c.view(B, C, 1, 1)                   # repondera canales

        # ---- Rama espacial ----
        mapa_avg = F_prima.mean(dim=1, keepdim=True)                 # (B, 1, H, W)
        mapa_max = F_prima.amax(dim=1, keepdim=True)                 # (B, 1, H, W)
        M_s = torch.sigmoid(self.conv(torch.cat([mapa_avg, mapa_max], dim=1)))
        return F_prima * M_s                                          # (B, C, H, W)


x_prueba = torch.randn(2, 256, 14, 14)   # forma de la salida de layer3 a 224px
cbam_didactico = CBAMDidactico(256)
print('entrada', tuple(x_prueba.shape), '-> salida', tuple(cbam_didactico(x_prueba).shape))

### 2.2 Verificación cruzada con la implementación del proyecto

Para comprobar que la implementación didáctica es exactamente la que se entrena, se
copian los pesos del CBAM del proyecto (`src/models/attention.py`) en el didáctico y se
verifica que ambas salidas coinciden.

In [ ]:
from src.models.attention import CBAM

torch.manual_seed(0)
cbam_proyecto = CBAM(256, reduction=16, spatial_kernel=7)
cbam_didactico.mlp.load_state_dict(cbam_proyecto.channel.mlp.state_dict())
cbam_didactico.conv.load_state_dict(cbam_proyecto.spatial.conv.state_dict())

with torch.no_grad():
    coincide_cbam = torch.allclose(cbam_didactico(x_prueba), cbam_proyecto(x_prueba), atol=1e-6)
print('CBAMDidactico coincide con src.models.attention.CBAM:', coincide_cbam)
assert coincide_cbam

### 2.3 Ubicación y costo del módulo

CBAM se inserta al final de `layer3` y `layer4`, las etapas profundas. Las etapas
tempranas codifican bordes y texturas de bajo nivel, donde reponderar aporta poco y los
mapas de características son grandes. La celda muestra dónde quedó el módulo y cuántos
parámetros añade.

In [ ]:
from src.models.classifier import build_classifier

modelo_con = build_classifier(cfg_cls, override_cbam=True)
modelo_sin = build_classifier(cfg_cls, override_cbam=False)

for etapa in ('layer3', 'layer4'):
    bloques = [type(m).__name__ for m in getattr(modelo_con.net, etapa).children()]
    print(f'{etapa} con CBAM: {bloques}')

n_con = sum(p.numel() for p in modelo_con.parameters())
n_sin = sum(p.numel() for p in modelo_sin.parameters())
n_cbam = sum(p.numel() for nombre, p in modelo_con.named_parameters() if 'channel' in nombre or 'spatial' in nombre)
print(f'\nsin CBAM: {n_sin:,} parametros')
print(f'con CBAM: {n_con:,} parametros  (+{n_con - n_sin:,}, un {100 * (n_con - n_sin) / n_sin:.2f}% mas)')
assert n_con - n_sin == n_cbam

### 2.4 Fine-tuning con tasas de aprendizaje diferenciadas

El backbone ya aprendió características visuales generales en ImageNet, y una tasa de
aprendizaje alta las destruiría. La capa de clasificación y los módulos CBAM, en cambio,
empiezan desde cero y necesitan avanzar más rápido. Por eso se definen dos grupos de
parámetros, con una tasa 10 veces mayor para los nuevos.

In [ ]:
grupos = modelo_con.param_groups(cfg_cls.train.lr, cfg_cls.train.head_lr_multiplier)
for nombre, grupo in zip(['backbone preentrenado', 'capa final + CBAM (nuevos)'], grupos):
    print(f"{nombre:28s} lr = {grupo['lr']:.0e}   parametros = {sum(p.numel() for p in grupo['params']):,}")
del modelo_con, modelo_sin

## 3. Entrenamiento

### 3.1 Pérdida ponderada por clase

Para que equivocarse en una clase minoritaria cueste más que equivocarse en `nv`, la
entropía cruzada usa un peso inverso a la frecuencia de cada clase en entrenamiento,
normalizado para que el promedio sea 1 y la escala de la pérdida no cambie:

$$w_c = \frac{N}{K\,n_c} \qquad\qquad \mathcal{L} = -\frac{\sum_i w_{y_i} \log p_{i,y_i}}{\sum_i w_{y_i}}$$

La celda calcula los pesos y los compara con la fórmula aplicada a mano.

In [ ]:
from src.eval.metrics import class_weights_balanced

conteo_train = splits[splits['split'] == 'train']['dx_idx'].value_counts().sort_index().to_numpy()
pesos = class_weights_balanced(conteo_train).numpy()

# Verificacion de la formula a mano
pesos_mano = conteo_train.sum() / (len(conteo_train) * conteo_train)
pesos_mano = pesos_mano / pesos_mano.mean()
assert np.allclose(pesos, pesos_mano, atol=1e-5)

print(pd.DataFrame({'imagenes_train': conteo_train, 'peso': pesos.round(3)}, index=CLASES))
print('\nverificado: los pesos coinciden con la formula')

### 3.2 El ciclo de entrenamiento

El script `src/train/train_classifier.py` repite en cada época el ciclo
`zero_grad → forward → pérdida → backward → step`. Sobre ese ciclo añade:

| Componente | Propósito |
| --- | --- |
| `AdamW` con weight decay | Optimizador con regularización desacoplada del gradiente adaptativo |
| `ReduceLROnPlateau(factor=0.5, patience=2)` | Si la pérdida de validación no mejora en 2 épocas, reduce la tasa de aprendizaje a la mitad |
| Precisión mixta (`float16`) | Reduce memoria y casi duplica la velocidad en la GPU |
| Checkpoint por macro-F1 de validación | Conserva el mejor modelo, no el de la última época |
| Mismo número de épocas, sin parada temprana | Las dos corridas completan las 25 épocas, así que la comparación no depende de cuándo se detuvo cada una |
| Estado de reanudación por época | Guarda en Drive el estado completo al terminar cada época, para continuar un entrenamiento interrumpido |

Antes de lanzar el entrenamiento completo, una verificación rápida: el ciclo, escrito
aquí paso a paso, debe reducir la pérdida sobre un lote fijo de 16 imágenes.

In [ ]:
from torch.utils.data import DataLoader, Subset

from src.data.datasets import make_dataloaders

_, datasets_cls = make_dataloaders(cfg_cls, 'classification')
lote_fijo = DataLoader(Subset(datasets_cls['train'], range(16)), batch_size=8, shuffle=False)

torch.manual_seed(0)
modelo = build_classifier(cfg_cls).to(DISPOSITIVO)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.AdamW(
    modelo.param_groups(cfg_cls.train.lr, cfg_cls.train.head_lr_multiplier),
    weight_decay=cfg_cls.train.weight_decay,
)

historial_prueba = []
modelo.train()
for paso in range(10):
    perdida_total = 0.0
    for imagenes, etiquetas in lote_fijo:
        imagenes, etiquetas = imagenes.to(DISPOSITIVO), etiquetas.to(DISPOSITIVO)
        optimizador.zero_grad()                  # 1. limpiar gradientes
        logits = modelo(imagenes)                # 2. forward
        perdida = criterio(logits, etiquetas)    # 3. perdida
        perdida.backward()                       # 4. backward
        optimizador.step()                       # 5. actualizar pesos
        perdida_total += perdida.item()
    historial_prueba.append(perdida_total / len(lote_fijo))

print('perdida por paso:', [round(p, 3) for p in historial_prueba])
assert historial_prueba[-1] < historial_prueba[0]
print('verificado: el ciclo reduce la perdida sobre un lote fijo')
del modelo, optimizador

### 3.3 Entrenar con CBAM (unos 40 minutos)

Cada época imprime la pérdida de entrenamiento y validación, la accuracy y el macro-F1 de
validación. Al terminar, evalúa el **mejor** checkpoint sobre test y guarda en Drive el
modelo, las métricas y la predicción de cada imagen de test. Si ya existe un resultado
con todas las épocas completas, la celda no vuelve a entrenar; si la sesión se
interrumpió a mitad del entrenamiento, al volver a ejecutarla continúa desde la última
época completada.

La función `entrenamiento_completo` hace esa comprobación. Un resultado con menos épocas
que las configuradas, o con alguna pérdida `NaN`, proviene de una corrida detenida antes de
tiempo o dañada: se conserva una copia con el sufijo `_incompleto` y se vuelve a entrenar.

In [ ]:
import shutil


def entrenamiento_completo(nombre_json):
    ruta = DIR_RESULTADOS / nombre_json
    if not ruta.exists():
        return False
    historial = leer_json(nombre_json)['history']
    finito = all(np.isfinite(h['train_loss']) and np.isfinite(h['val_loss']) for h in historial)
    if len(historial) >= cfg_cls.train.epochs and finito:
        print(f'Ya entrenado ({len(historial)} epocas): se usa el resultado guardado en Drive.')
        return True
    copia = ruta.with_name(ruta.stem + '_incompleto.json')
    if not copia.exists():
        shutil.copy(ruta, copia)
    motivo = 'tiene perdidas NaN' if not finito else f'tiene {len(historial)} de {cfg_cls.train.epochs} epocas'
    print(f'{nombre_json} {motivo}: se vuelve a entrenar (copia del resultado anterior en {copia.name}).')
    return False


if not entrenamiento_completo('classification_cbam.json'):
    !python -m src.train.train_classifier --config configs/classification.yaml

### 3.4 Entrenar sin CBAM: comparación controlada (unos 40 minutos)

Para atribuir la diferencia al módulo CBAM, la segunda corrida solo puede diferir en ese
módulo. Por eso comparte con la primera la semilla, la partición, las aumentaciones, los
hiperparámetros y el número de épocas; las capas comunes arrancan con los mismos pesos
iniciales y los lotes llegan en el mismo orden.

El número de épocas importa: una primera corrida sin CBAM con parada temprana tuvo una
inestabilidad en la época 8 y se detuvo en la 12, mientras la corrida con CBAM completó
las 25. Esa diferencia de entrenamiento se confundía con el efecto de CBAM, por lo que
ambas corridas se hacen ahora con las 25 épocas completas.

In [ ]:
if not entrenamiento_completo('classification_nocbam.json'):
    !python -m src.train.train_classifier --config configs/classification.yaml --no-cbam

### 3.5 Curvas de entrenamiento

La pérdida de validación es la señal más directa de sobreajuste: si baja y luego sube
mientras la de entrenamiento sigue bajando, el modelo empezó a memorizar. Se grafican
ambas pérdidas y el macro-F1 de validación de las dos corridas.

In [ ]:
res_con = leer_json('classification_cbam.json')
res_sin = leer_json('classification_nocbam.json')
COLORES = {'sin CBAM': '#DD8452', 'con CBAM': '#4C72B0'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for res, nombre in [(res_sin, 'sin CBAM'), (res_con, 'con CBAM')]:
    h = pd.DataFrame(res['history'])
    axes[0].plot(h['epoch'], h['train_loss'], '--', color=COLORES[nombre], label=f'train ({nombre})')
    axes[0].plot(h['epoch'], h['val_loss'], '-', color=COLORES[nombre], label=f'validacion ({nombre})')
    axes[1].plot(h['epoch'], h['val_macro_f1'], '-o', markersize=3, color=COLORES[nombre], label=nombre)
axes[0].set_title('Perdida: entrenamiento vs validacion')
axes[1].set_title('Macro-F1 en validacion')
for ax in axes:
    ax.set_xlabel('epoca')
    ax.legend()
plt.tight_layout()
guardar_figura(fig, 'curvas_clasificador')
plt.show()

for res, nombre in [(res_sin, 'sin CBAM'), (res_con, 'con CBAM')]:
    h = pd.DataFrame(res['history'])
    mejor = h.loc[h['val_macro_f1'].idxmax()]
    print(f"{nombre}: {len(h)} epocas en {res['train_minutes']:.0f} min, "
          f"mejor macro-F1 de validacion {mejor['val_macro_f1']:.4f} en la epoca {int(mejor['epoch'])}")

## 4. Resultados en test

Se consolidan las métricas de las dos corridas en una tabla comparativa. El mismo script
calcula además, sobre **todas** las imágenes de test, qué fracción del mapa de Grad-CAM de
cada modelo cae dentro de la lesión anotada (sección 5), por lo que tarda unos minutos.

In [ ]:
!python -m src.eval.compare_ablation --cls-config configs/classification.yaml

Tablas de resultados:

- **Comparación con y sin CBAM:** parámetros, tamaño, accuracy, accuracy balanceada y
  macro-F1.
- **Localización de la atención:** media y mediana sobre todas las imágenes, promedio
  por clase (cada clase pesa igual, como en el macro-F1), diferencia media frente al
  modelo sin CBAM y porcentaje de imágenes en que CBAM concentra más atención en la lesión.
- **Localización por clase:** para detectar si el efecto se concentra en pocas clases.

In [ ]:
tabla_cbam = pd.read_csv(DIR_RESULTADOS / 'ablacion_cbam.csv').set_index('modelo')
tabla_localizacion = pd.read_csv(DIR_RESULTADOS / 'localizacion_atencion.csv').set_index('modelo')
tabla_localizacion_clase = pd.read_csv(DIR_RESULTADOS / 'localizacion_atencion_por_clase.csv').set_index('clase')
display(tabla_cbam[['params_M', 'tamano_MB', 'accuracy', 'balanced_acc', 'macro_F1']])
display(tabla_localizacion)
display(tabla_localizacion_clase)

### 4.1 ¿La diferencia supera al azar del conjunto de test?

El test tiene 1.502 imágenes, pero solo 19 de `df` y 21 de `vasc`: acertar o fallar unas
pocas cambia bastante su F1 y, con él, el macro-F1. Para estimar cuánto de la diferencia
entre los dos modelos podría deberse a qué imágenes quedaron en test se usa un
**bootstrap pareado** (Efron y Tibshirani, 1993) sobre las predicciones guardadas:

1. Se sortean 1.502 imágenes del test con reemplazo.
2. Sobre **las mismas** imágenes se calcula la métrica de los dos modelos y su diferencia.
3. Se repite 10.000 veces.

El intervalo entre los percentiles 2,5 y 97,5 de las diferencias es un intervalo de
confianza del 95%: si no contiene el 0, la diferencia no se explica por la composición
del test. La prueba no mide la variación entre entrenamientos con distintas semillas.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score

pred_con, pred_sin = res_con['test_predictions'], res_sin['test_predictions']
assert pred_con['image_id'] == pred_sin['image_id']
y_real = np.array(pred_con['y_true'])
y_con, y_sin = np.array(pred_con['y_pred']), np.array(pred_sin['y_pred'])
etiquetas = list(range(len(CLASES)))

metricas_bootstrap = {
    'macro_F1': lambda t, p: f1_score(t, p, labels=etiquetas, average='macro', zero_division=0),
    'balanced_acc': balanced_accuracy_score,
    'accuracy': lambda t, p: (t == p).mean(),
}
generador = np.random.default_rng(42)
n_test, n_sorteos = len(y_real), 10_000
diferencias = {m: np.empty(n_sorteos) for m in metricas_bootstrap}
for i in range(n_sorteos):
    idx = generador.integers(0, n_test, n_test)
    for m, f in metricas_bootstrap.items():
        diferencias[m][i] = f(y_real[idx], y_con[idx]) - f(y_real[idx], y_sin[idx])

filas = []
for m, f in metricas_bootstrap.items():
    bajo, alto = np.percentile(diferencias[m], [2.5, 97.5])
    filas.append({'metrica': m, 'con CBAM': f(y_real, y_con), 'sin CBAM': f(y_real, y_sin),
                  'diferencia': f(y_real, y_con) - f(y_real, y_sin), 'IC95 inferior': bajo,
                  'IC95 superior': alto, '% sorteos en que gana CBAM': 100 * (diferencias[m] > 0).mean()})
tabla_bootstrap = pd.DataFrame(filas).set_index('metrica')
tabla_bootstrap.to_csv(DIR_RESULTADOS / 'ablacion_cbam_bootstrap.csv')
display(tabla_bootstrap.round(4))

solo_con = int(((y_con == y_real) & (y_sin != y_real)).sum())
solo_sin = int(((y_sin == y_real) & (y_con != y_real)).sum())
print(f'imagenes que solo acierta el modelo con CBAM: {solo_con} | solo sin CBAM: {solo_sin}')

### 4.2 Matrices de confusión

Los números son conteos; el color es la proporción de cada fila, es decir, el recall por
clase. La fila `mel` es la de mayor relevancia clínica: muestra cuántos melanomas se
confunden con otras clases, en particular con `nv`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, res, nombre in zip(axes, [res_sin, res_con], ['sin CBAM', 'con CBAM']):
    cm = np.array(res['test']['confusion_matrix'])
    totales = cm.sum(axis=1, keepdims=True)
    cm_proporcion = np.divide(cm, totales, out=np.zeros(cm.shape), where=totales > 0)
    ax.imshow(cm_proporcion, cmap='Blues', vmin=0, vmax=1)
    for i in range(len(CLASES)):
        for j in range(len(CLASES)):
            color = 'white' if cm_proporcion[i, j] > 0.5 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8, color=color)
    ax.set_xticks(range(len(CLASES)), CLASES, rotation=45)
    ax.set_yticks(range(len(CLASES)), CLASES)
    ax.set_xlabel('prediccion')
    ax.set_ylabel('real')
    ax.set_title(f"{nombre}  (macro-F1 = {res['test']['macro_f1']:.3f})")
plt.tight_layout()
guardar_figura(fig, 'matrices_confusion')
plt.show()

### 4.3 F1 por clase

Permite ver si el efecto de CBAM es homogéneo o se concentra en pocas clases. Las
clases `df` y `vasc` tienen unas 20 imágenes de test, así que acertar o fallar dos
imágenes cambia mucho su F1: sus diferencias deben interpretarse con cautela.

In [ ]:
f1_por_clase = pd.DataFrame({
    'sin CBAM': res_sin['test']['per_class_f1'],
    'con CBAM': res_con['test']['per_class_f1'],
}).reindex(CLASES)
f1_por_clase['diferencia'] = f1_por_clase['con CBAM'] - f1_por_clase['sin CBAM']
f1_por_clase['imagenes_test'] = splits[splits['split'] == 'test']['dx'].value_counts().reindex(CLASES).fillna(0).astype(int)
display(f1_por_clase.round(3))

ax = f1_por_clase[['sin CBAM', 'con CBAM']].plot.bar(
    figsize=(9, 3.5), rot=0, color=[COLORES['sin CBAM'], COLORES['con CBAM']]
)
ax.set_ylim(0, 1)
ax.set_ylabel('F1 en test')
ax.set_title('F1 por clase, con y sin CBAM')
plt.tight_layout()
guardar_figura(ax.get_figure(), 'f1_por_clase')
plt.show()

## 5. Grad-CAM: ¿en qué se fija el clasificador?

### 5.1 Formulación

Grad-CAM (Selvaraju et al., 2017) toma, para la clase predicha $c$, las activaciones
$A^k$ de la última etapa convolucional (`layer4`) y el gradiente del logit $y^c$
respecto a ellas. El promedio espacial del gradiente mide cuánto aporta cada canal $k$ a
la clase:

$$\alpha_k^c = \frac{1}{Z}\sum_{i}\sum_{j} \frac{\partial y^c}{\partial A^k_{ij}}$$

El mapa es la combinación de los canales ponderada por esos pesos, y el ReLU conserva
solo la evidencia **a favor** de la clase:

$$L^c = \text{ReLU}\Big(\sum_k \alpha_k^c\, A^k\Big)$$

La celda siguiente lo implementa paso a paso, con *hooks* de PyTorch para capturar las
activaciones y su gradiente.

In [ ]:
import torch.nn.functional as F


def grad_cam_didactico(modelo, capa, x, clase=None):
    '''Grad-CAM paso a paso. x: una imagen normalizada, forma (1, 3, H, W).'''
    guardado = {}

    def guardar_gradiente(grad):
        guardado['dA'] = grad                                    # dy^c / dA^k

    def guardar_activacion(modulo, entrada, salida):
        guardado['A'] = salida                                   # A^k
        salida.register_hook(guardar_gradiente)

    gancho = capa.register_forward_hook(guardar_activacion)
    modelo.eval()
    logits = modelo(x)
    gancho.remove()

    if clase is None:
        clase = int(logits.argmax(dim=1))
    modelo.zero_grad()
    logits[0, clase].backward()                                  # gradiente del logit y^c

    A, dA = guardado['A'].detach(), guardado['dA'].detach()      # (1, K, h, w)
    alpha = dA.mean(dim=(2, 3), keepdim=True)                    # alpha_k: promedio espacial
    mapa = F.relu((alpha * A).sum(dim=1, keepdim=True))          # ReLU(sum_k alpha_k A^k)
    mapa = F.interpolate(mapa, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]

    mapa = mapa - mapa.min()                                     # normalizar a [0, 1]
    if mapa.max() > 1e-8:
        mapa = mapa / mapa.max()
    return mapa.cpu().numpy(), clase

### 5.2 Verificación cruzada con la implementación del proyecto

Se cargan los dos clasificadores entrenados y se compara el mapa de la implementación
didáctica con el de `src/explain/gradcam.py` sobre una imagen real de test.

In [ ]:
from src.data import transforms as T
from src.data.datasets import read_mask, read_rgb
from src.explain.gradcam import GradCAM, attention_mass_in_mask, overlay_heatmap


def cargar_clasificador(con_cbam):
    etiqueta = 'cbam' if con_cbam else 'nocbam'
    modelo = build_classifier(cfg_cls, override_cbam=con_cbam)
    estado = torch.load(DIR_MODELOS / f'classifier_{etiqueta}_best.pt', map_location=DISPOSITIVO)
    modelo.load_state_dict(estado['model'])
    return modelo.to(DISPOSITIVO).eval()


clasificador_con = cargar_clasificador(True)
clasificador_sin = cargar_clasificador(False)
tf_eval = T.classification_eval_transform(cfg_cls)

fila = splits[splits['split'] == 'test'].iloc[0]
x = tf_eval(read_rgb(fila['image_path'])).unsqueeze(0).to(DISPOSITIVO)

mapa_didactico, clase_didactico = grad_cam_didactico(clasificador_con, clasificador_con.gradcam_target_layer, x)
with GradCAM(clasificador_con, clasificador_con.gradcam_target_layer) as gradcam:
    mapa_proyecto, clase_proyecto = gradcam(x)

coincide_gradcam = clase_didactico == clase_proyecto and np.allclose(mapa_didactico, mapa_proyecto, atol=1e-4)
print('grad_cam_didactico coincide con src.explain.gradcam.GradCAM:', coincide_gradcam)
assert coincide_gradcam

### 5.3 Mapas con y sin CBAM sobre imágenes de test

Para cuatro imágenes de test se muestra el borde anotado de la lesión y el mapa de
Grad-CAM de cada modelo, junto con la fracción del mapa que cae dentro de la lesión. Si
CBAM cumple su función, el mapa debería concentrarse más en la lesión y menos en vello,
burbujas o los bordes oscuros de la imagen. La evidencia cuantitativa es la tabla de
localización de la sección 4; estas figuras son ejemplos ilustrativos.

Esta medida usa la máscara **anotada** del dataset. El porcentaje que muestra el
aplicativo es distinto: usa la máscara **predicha** por el segmentador,
porque en uso real no hay anotación, así que ambos números no deben compararse.

In [ ]:
from src.explain.isolate import isolate_lesion

test_con_mascara = splits[(splits['split'] == 'test') & splits['has_mask']]
ejemplos = [test_con_mascara[test_con_mascara['dx'] == c].iloc[0]
            for c in ['mel', 'bcc', 'bkl', 'nv'] if (test_con_mascara['dx'] == c).any()]

fig, axes = plt.subplots(len(ejemplos), 3, figsize=(10, 3.4 * len(ejemplos)), squeeze=False)
for i, fila in enumerate(ejemplos):
    imagen_pil = read_rgb(fila['image_path'])
    imagen = np.array(imagen_pil.resize((224, 224)))
    mascara = np.array(Image.fromarray(read_mask(fila['mask_path']) * 255).resize((224, 224), Image.NEAREST)) > 127
    x = tf_eval(imagen_pil).unsqueeze(0).to(DISPOSITIVO)

    borde_real = isolate_lesion(imagen, mascara.astype(np.float32), morph_close_kernel=0, keep_largest_component=False)
    axes[i, 0].imshow(borde_real.overlay)
    axes[i, 0].set_title(f"real: {fila['dx']} (borde anotado)", fontsize=9)

    for j, (modelo, nombre) in enumerate([(clasificador_sin, 'sin CBAM'), (clasificador_con, 'con CBAM')], start=1):
        with GradCAM(modelo, modelo.gradcam_target_layer) as gradcam:
            mapa, clase = gradcam(x)
        masa = attention_mass_in_mask(mapa, mascara.astype(np.float32))
        axes[i, j].imshow(overlay_heatmap(imagen, mapa))
        axes[i, j].set_title(f'{nombre}: predice {CLASES[clase]}\n{masa:.0%} del mapa dentro de la lesion', fontsize=9)

for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
guardar_figura(fig, 'gradcam_con_sin_cbam')
plt.show()

## 6. Optimización: cuantización INT8

La cuantización representa los pesos y las activaciones con enteros de 8 bits en lugar
de números de punto flotante de 32 bits (Jacob et al., 2018). Se espera un modelo cerca
de cuatro veces más pequeño y más rápido en CPU, a cambio de una posible pérdida de
exactitud que hay que medir. El proceso tiene tres pasos:

1. **Exportación** del clasificador con CBAM al formato ONNX.
2. **Cuantización estática** con calibración: se pasan 200 imágenes reales de
   entrenamiento para estimar el rango de las activaciones y fijar las escalas. Imágenes
   no representativas producirían escalas mal calibradas.
3. **Evaluación** de las versiones FP32 e INT8 sobre el mismo conjunto de test: tamaño en
   disco, accuracy y macro-F1.

Si la caída de macro-F1 supera la tolerancia definida en la configuración
(`max_f1_drop` = 0,02), la cuantización se considera inaceptable. El modelo INT8 se
evalúa en CPU, por lo que esta celda tarda unos minutos aunque haya GPU.

In [ ]:
!python -m src.optimize.quantize --config configs/classification.yaml

Comparación antes y después de la cuantización: tamaño, métricas, porcentaje de
reducción y caída de macro-F1 frente a la tolerancia.

In [ ]:
opt = leer_json('optimization.json')
tabla_opt = pd.DataFrame({
    version: {'tamano_MB': opt[clave]['size_mb'], 'accuracy': opt[clave]['accuracy'], 'macro_F1': opt[clave]['macro_f1']}
    for version, clave in [('FP32 (ONNX)', 'fp32'), ('INT8 (ONNX)', 'int8')]
}).T.round(4)
display(tabla_opt)

print(f"reduccion de tamano: {opt['size_reduction']:.1%}")
estado = 'dentro' if opt['within_tolerance'] else 'FUERA'
print(f"caida de macro-F1: {opt['macro_f1_drop']:+.4f} (tolerancia {cfg_cls.optimize.max_f1_drop}): {estado} de la tolerancia")

## 7. Tiempos de inferencia en GPU

Se mide el tiempo de inferencia de los cuatro modelos del proyecto (clasificador con y
sin CBAM, segmentador con y sin self-attention), con lotes de 1 y 8 imágenes. Los valores
de los pesos no influyen en el tiempo, así que no es necesario haber entrenado el
segmentador.

Para que la medición sea fiable se calienta el modelo antes de medir, se sincroniza la
GPU antes de detener el cronómetro y se reportan la mediana y el percentil 95 además del
promedio. El hardware queda registrado con cada medición. La medición en CPU se realiza
en un equipo local y se consolida en el mismo archivo.

In [ ]:
!python -m src.eval.benchmark --device cuda --batch-sizes 1 8

Tabla con la medición de GPU más reciente: mediana, percentil 95, milisegundos por imagen
e imágenes por segundo.

In [ ]:
tiempos = pd.read_csv(DIR_RESULTADOS / 'timings.csv')
tiempos_gpu = tiempos[tiempos['device'] == 'cuda']
tiempos_gpu = tiempos_gpu[tiempos_gpu['timestamp'] == tiempos_gpu['timestamp'].max()]   # la medicion mas reciente
display(tiempos_gpu[['model', 'gpu', 'batch_size', 'p50_ms', 'p95_ms', 'ms_per_image', 'fps']].round(2))

## 8. Checklist de verificación

Resumen de las verificaciones del notebook y de los archivos que deben haber quedado en
Drive. Si alguna falla, el notebook no está completo.

In [ ]:
print('=' * 70)
print('CHECKLIST DE VERIFICACION - Notebook 01, clasificador')
print('=' * 70)

verificaciones = [
    ('Ninguna lesion aparece en mas de un split', fuga == 0),
    ('CBAMDidactico coincide con la implementacion del proyecto', bool(coincide_cbam)),
    ('El ciclo de entrenamiento reduce la perdida', historial_prueba[-1] < historial_prueba[0]),
    ('Grad-CAM didactico coincide con la implementacion del proyecto', bool(coincide_gradcam)),
    ('Con CBAM supera en macro-F1 al clasificador constante', res_con['test']['macro_f1'] > f1_constante),
    ('Sin CBAM supera en macro-F1 al clasificador constante', res_sin['test']['macro_f1'] > f1_constante),
    ('Las dos corridas completaron el mismo numero de epocas',
     len(res_con['history']) == len(res_sin['history']) == cfg_cls.train.epochs),
    ('El modelo INT8 pesa menos que el FP32', opt['int8']['size_mb'] < opt['fp32']['size_mb']),
]
for nombre in ['classifier_cbam_best.pt', 'classifier_nocbam_best.pt', 'classifier_fp32.onnx', 'classifier_int8.onnx']:
    verificaciones.append((f'Guardado en Drive: models/{nombre}', (DIR_MODELOS / nombre).exists()))
for nombre in ['classification_cbam.json', 'classification_nocbam.json', 'ablacion_cbam.csv',
               'ablacion_cbam_bootstrap.csv',
               'localizacion_atencion.csv', 'localizacion_atencion_por_clase.csv',
               'optimization.json', 'timings.csv']:
    verificaciones.append((f'Guardado en Drive: results/{nombre}', (DIR_RESULTADOS / nombre).exists()))

for descripcion, resultado in verificaciones:
    print(f"[{'PASA' if resultado else 'FALLA'}] {descripcion}")

todo_ok = all(resultado for _, resultado in verificaciones)
print('=' * 70)
print('RESULTADO FINAL:', 'TODAS LAS VERIFICACIONES PASAN' if todo_ok else 'HAY VERIFICACIONES FALLIDAS')
assert todo_ok
print('=' * 70)

### Cierre

Quedaron en Drive, dentro de la carpeta `dermascope`:

- `models/`: los dos clasificadores entrenados y sus versiones ONNX FP32 e INT8.
- `reports/results/`: métricas de test, predicciones por imagen, tabla comparativa con y
  sin CBAM, localización de Grad-CAM, resultados de la cuantización y tiempos en GPU.
- `reports/figures/`: las figuras del notebook, listas para el informe.

El siguiente paso es `02_entrenar_segmentador_colab.ipynb`, que puede ejecutarse en otra
sesión de Colab: usa la misma partición y la misma carpeta de Drive.

## Referencias

- Codella, N., Rotemberg, V., Tschandl, P., et al. (2019). *Skin lesion analysis toward melanoma detection 2018: A challenge hosted by the International Skin Imaging Collaboration (ISIC)*. arXiv:1902.03368.
- Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall.
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*.
- Hu, J., Shen, L., & Sun, G. (2018). Squeeze-and-excitation networks. *CVPR*.
- Jacob, B., Kligys, S., Chen, B., et al. (2018). Quantization and training of neural networks for efficient integer-arithmetic-only inference. *CVPR*.
- Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
- Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. *ICCV*.
- Tschandl, P., Rosendahl, C., & Kittler, H. (2018). The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. *Scientific Data*, 5, 180161.
- Woo, S., Park, J., Lee, J.-Y., & Kweon, I. S. (2018). CBAM: Convolutional block attention module. *ECCV*.